In [1]:
!pip install -q datasets==2.20.0 scikit-learn
import torch
print("GPU available:", torch.cuda.is_available())  # must print True

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 21.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.5.0 which is incompatible.
GPU available: True


In [2]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = ds["train"].to_pandas()
print(df.shape)
print(df.columns.tolist())
df = df[["instruction", "intent", "response"]].dropna()
print("Number of intents:", df["intent"].nunique())
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

(26872, 5)
['flags', 'instruction', 'category', 'intent', 'response']
Number of intents: 27


,instruction,intent,response
0,question about cancelling order {{Order Number}},cancel_order,I've understood you have a question regarding ...
1,i have a question about cancelling oorder {{Or...,cancel_order,I've been informed that you have a question ab...
2,i need help cancelling puchase {{Order Number}},cancel_order,I can sense that you're seeking assistance wit...
3,I need to cancel purchase {{Order Number}},cancel_order,I understood that you need assistance with can...
4,"I cannot afford this order, cancel purchase {{...",cancel_order,I'm sensitive to the fact that you're facing f...


In [3]:
import re

def clean(t):
    t = str(t).lower()
    t = re.sub(r"\{\{.*?\}\}", " placeholder ", t)   # bitext uses {{Order Number}} etc.
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df["text"] = df["instruction"].apply(clean)
df = df[df["text"].str.len() > 0]

# one representative suggested reply per intent
response_map = df.groupby("intent")["response"].first().to_dict()

# intents that should trigger human escalation (edit names to match your data if needed)
print(sorted(df["intent"].unique()))
ESCALATE = {"complaint", "review", "contact_human_agent"}  # tweak after seeing the list above
print(len(df), "usable rows")

['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoice', 'check_payment_methods', 'check_refund_policy', 'complaint', 'contact_customer_service', 'contact_human_agent', 'create_account', 'delete_account', 'delivery_options', 'delivery_period', 'edit_account', 'get_invoice', 'get_refund', 'newsletter_subscription', 'payment_issue', 'place_order', 'recover_password', 'registration_problems', 'review', 'set_up_shipping_address', 'switch_account', 'track_order', 'track_refund']
26872 usable rows


In [4]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["intent"], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df["intent"], random_state=42)
print("train", len(train_df), "val", len(val_df), "test", len(test_df))

train 18810 val 4031 test 4031


In [5]:
from collections import Counter
import numpy as np

PAD, UNK, MAX_LEN = 0, 1, 30

counter = Counter(w for t in train_df["text"] for w in t.split())
vocab = {"<PAD>": 0, "<UNK>": 1}
for w, c in counter.items():
    if c >= 2:                       # ignore very rare words
        vocab[w] = len(vocab)

labels = sorted(df["intent"].unique())
label_map = {l: i for i, l in enumerate(labels)}
inv_label = {i: l for l, i in label_map.items()}

def encode(t):
    ids = [vocab.get(w, UNK) for w in t.split()][:MAX_LEN]
    ids += [PAD] * (MAX_LEN - len(ids))
    return ids

def to_tensors(frame):
    X = torch.tensor([encode(t) for t in frame["text"]], dtype=torch.long)
    y = torch.tensor([label_map[i] for i in frame["intent"]], dtype=torch.long)
    return X, y

Xtr, ytr = to_tensors(train_df)
Xva, yva = to_tensors(val_df)
Xte, yte = to_tensors(test_df)
print("vocab size", len(vocab), "| classes", len(label_map))

vocab size 979 | classes 27


In [6]:
import torch.nn as nn

class CNNBiLSTM(nn.Module):
    def __init__(self, vocab_size, n_classes, emb=128, hid=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb, 100, k, padding=k//2) for k in (2, 3, 4)])
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.4)
        self.fc = nn.Linear(100*3 + hid*2, n_classes)

    def forward(self, x):
        e = self.emb(x)                                  # B, L, E
        c = e.transpose(1, 2)                            # B, E, L  (for Conv1d)
        cnn = [torch.relu(conv(c)).max(dim=2).values for conv in self.convs]
        cnn = torch.cat(cnn, dim=1)                      # phrase features
        out, _ = self.lstm(e)
        lstm = out.mean(dim=1)                           # context features
        z = self.drop(torch.cat([cnn, lstm], dim=1))
        return self.fc(z)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CNNBiLSTM(len(vocab), len(label_map)).to(device)
print(model)

CNNBiLSTM(
  (emb): Embedding(979, 128, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(128, 100, kernel_size=(2,), stride=(1,), padding=(1,))
    (1): Conv1d(128, 100, kernel_size=(3,), stride=(1,), padding=(1,))
    (2): Conv1d(128, 100, kernel_size=(4,), stride=(1,), padding=(2,))
  )
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (drop): Dropout(p=0.4, inplace=False)
  (fc): Linear(in_features=556, out_features=27, bias=True)
)


In [7]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss()

def evaluate(X, y):
    model.eval()
    with torch.no_grad():
        preds = model(X.to(device)).argmax(1).cpu()
    return accuracy_score(y, preds), f1_score(y, preds, average="macro")

best_f1 = 0.0
EPOCHS = 12
for ep in range(1, EPOCHS+1):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = lossf(model(xb), yb)
        loss.backward()
        opt.step()
    acc, f1 = evaluate(Xva, yva)
    print(f"epoch {ep:2d} | val acc {acc:.3f} | val macroF1 {f1:.3f}")
    if f1 >= best_f1:                       # <-- checkpoint rule from the spec
        best_f1 = f1
        torch.save(model.state_dict(), "best_cnn_bilstm_intent.pth")
print("Best val macro-F1:", round(best_f1, 3))

epoch  1 | val acc 0.980 | val macroF1 0.980
epoch  2 | val acc 0.992 | val macroF1 0.992
epoch  3 | val acc 0.991 | val macroF1 0.991
epoch  4 | val acc 0.993 | val macroF1 0.993
epoch  5 | val acc 0.992 | val macroF1 0.992
epoch  6 | val acc 0.994 | val macroF1 0.994
epoch  7 | val acc 0.992 | val macroF1 0.992
epoch  8 | val acc 0.994 | val macroF1 0.994
epoch  9 | val acc 0.994 | val macroF1 0.994
epoch 10 | val acc 0.993 | val macroF1 0.993
epoch 11 | val acc 0.993 | val macroF1 0.993
epoch 12 | val acc 0.993 | val macroF1 0.993
Best val macro-F1: 0.994


In [8]:
model.load_state_dict(torch.load("best_cnn_bilstm_intent.pth"))
acc, f1 = evaluate(Xte, yte)
print(f"TEST accuracy {acc:.3f} | TEST macro-F1 {f1:.3f}")

TEST accuracy 0.996 | TEST macro-F1 0.996


In [9]:
import json

config = {
    "vocab_size": len(vocab), "n_classes": len(label_map),
    "max_len": MAX_LEN, "emb": 128, "hid": 128,
    "escalate_intents": sorted(ESCALATE),
    "low_confidence_threshold": 0.40
}
json.dump(vocab,        open("vocab.json", "w"))
json.dump(label_map,    open("label_map.json", "w"))
json.dump(response_map, open("response_map.json", "w"))
json.dump(config,       open("model_config.json", "w"))
print("Artifacts saved.")

Artifacts saved.


In [10]:
from google.colab import files
!zip artifacts.zip best_cnn_bilstm_intent.pth vocab.json label_map.json response_map.json model_config.json
files.download("artifacts.zip")

  adding: best_cnn_bilstm_intent.pth (deflated 8%)
  adding: vocab.json (deflated 60%)
  adding: label_map.json (deflated 52%)
  adding: response_map.json (deflated 65%)
  adding: model_config.json (deflated 24%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>